# Designing a two-level thermometer
A thermometer estimates a bath temperature by counting excitations of a small system with a calibrated gap. Select a gap, generate a repeatable data set and determine whether a count of zero means zero temperature.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

temperature_K = 4.0
gap_K = 9.60       # gap energy divided by Boltzmann's constant
trials = 400
seed = 21
assert temperature_K>0 and gap_K>0 and trials>=10

## Derive the likelihood from the canonical occupations
For independent preparations the event count follows a binomial law. We use a stable logarithm for each outcome to compare temperatures, including when one outcome was never recorded.

In [ ]:
def excited(x):
    w=np.exp(-np.asarray(x))
    return w/(1+w)
x=gap_K/temperature_K
p=excited(x); k=np.random.default_rng(seed).binomial(trials,p)
def loglik(T):
    u=gap_K/np.asarray(T)
    return -k*u-trials*np.logaddexp(0,-u)
lower=1/(x*np.sqrt(trials*p*(1-p))) if p>0 else np.inf
estimate=gap_K/np.log((trials-k)/k) if 0<k<trials/2 else (0. if k==0 else np.inf)
print(f"Excited fraction {p:.5f}, observed {k}/{trials}, positive-temperature MLE {estimate:.5g} K")
print(f"Local relative standard-error bound {lower:.4g}")

## Compare the probability, thermal response and experimental likelihood
The sensitivity of an energy reading is governed by its variance together with the rate at which its mean changes. A gap that is very large gives almost identical readings on every trial. A very small gap produces many transitions but their probability barely changes with temperature.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.9),layout="constrained")
u=np.linspace(.01,max(8,1.1*x),400); probs=excited(u)
ax[0].plot(u,probs,color="#375b9a",label="Excited probability")
ax[0].plot(u,u*u*probs*(1-probs),color="#175868",label="C / kB")
ax[0].axvline(x,ls=":",color="#a23d21",label="Selected gap")
ax[0].set(xlabel="Gap / kBT",ylabel="Dimensionless response",title="Choose a gap near the peak");ax[0].legend(fontsize=8)
T=np.geomspace(temperature_K*.25,temperature_K*4,350); ll=loglik(T)
ax[1].semilogx(T,np.exp(ll-ll.max()),color="#175868")
ax[1].axvline(temperature_K,ls=":",color="#375b9a",label="Preparation temperature")
if np.isfinite(estimate) and estimate>0: ax[1].axvline(estimate,color="#a23d21",ls="--",label="Fitted temperature")
ax[1].set(xlabel="Trial temperature (K)",ylabel="Likelihood / maximum on grid",title=f"One record with {k} excited sites");ax[1].legend(fontsize=8)
plt.show()

## Find the optimum and inspect a boundary outcome
The optimum solves x tanh(x/2) = 2. Bisection uses the monotonicity on the positive axis. If no excitations were seen, calculate an exact one-sided 95% upper limit on the excitation probability, then invert it if that limit lies below one half.

In [ ]:
a,b=1.,4.
for _ in range(50):
    m=(a+b)/2
    if m*np.tanh(m/2)>2:b=m
    else:a=m
opt=(a+b)/2
p95=-np.expm1(np.log(.05)/trials)
T95=gap_K/np.log((1-p95)/p95) if p95<.5 else np.inf
print(f"Optimal gap/kBT {opt:.6f}. For zero successes an upper confidence limit is {T95:.5g} K.")
assert np.isclose(opt,2.39935728,rtol=1e-7)
assert np.isclose((1-p95)**trials,.05)